### TF-IDF + Ridge vs BERT Baseline Comparison

Performance comparison on multilabel reappraisal strategy classification.

The TF-IDF notebook evaluates binary labels at vote_fraction >= 0.6
BERT target scheme is majority, defined as more than half of valid annotators marking the strategy present

BERT also reports weak_presence (at least 1) and strong_presence (80%).

1. SITUATION + REAPPRAISAL predicting the eight strategy labels (`GM`, `PR`, `A`, `EJ`, `N`, `UC`, `V`, `H`)
2. Soft-label prediction & binary classification (Just to visualize). Ridge and BERT trained on annotator vote fractions (e.g. 3 out of 5) binary metrics depend on different thresholds.
3. micro F1 emphasizes frequent labels, macro F1 gives each label equal weight, samples F1 measures per-example multilabel quality, hamming loss rewards correct negatives, average precision evaluates ranking, and MAE/correlation evaluate soft-label quality

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_dir() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "Code").is_dir() and (candidate / "README.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root; start Jupyter from inside the repository.")

ROOT = find_project_dir()
BERT_DIR = ROOT / "Code" / "BERT_models"
TFIDF_OUTPUT_DIR = ROOT / "Code" / "TF_IDF" / "outputs"

LABELS = ["GM", "PR", "A", "EJ", "N", "UC", "V", "H"]
STRATEGY_NAMES = {
    "GM": "growth mindset",
    "PR": "positive repurposing",
    "A": "aberration",
    "EJ": "external justification",
    "N": "normalizing",
    "UC": "unwarranted conclusions",
    "V": "validation",
    "H": "humor",
}

plt.style.use("default")

TF-IDF uses word n-grams with Ridge regression on annotator vote fractions, then tunes per-label thresholds on the validation split. The prevalence baseline is kept as a simple reference.

The BERT experiments fit transformer encoders with binomial negative log-likelihood against soft annotator vote fractions. It reports soft-label metrics and binary metrics( weak_presence , majority and strong_presence)


In [ ]:
model_setup = pd.DataFrame([
    {
        "family": "Prevalence",
        "model": "Prevalence baseline",
        "representation": "training-label prevalence",
        "training_target": "binary label frequencies",
        "binary_target_for_headline": "vote_fraction >= 0.6",
        "thresholding": "fixed 0.5 threshold on prevalence scores",
        "source": "Code/TF_IDF/outputs/tfidf_model_summary.csv",
    },
    {
        "family": "TF-IDF",
        "model": "TF-IDF + tuned Ridge",
        "representation": "word TF-IDF n-grams",
        "training_target": "annotator vote fractions",
        "binary_target_for_headline": "vote_fraction >= 0.6",
        "thresholding": "per-label validation-tuned thresholds",
        "source": "Code/TF_IDF/TFIDF_Ridge_Baseline.ipynb",
    },
    {
        "family": "BERT",
        "model": "bert-base-uncased",
        "representation": "Transformer encoder",
        "training_target": "annotator vote fractions / binomial NLL",
        "binary_target_for_headline": "majority scheme",
        "thresholding": "per-label validation-tuned thresholds",
        "source": "Code/BERT_models/bert_base_uncased_random_seed42_metrics.json",
    },
    {
        "family": "BERT",
        "model": "bert-large-uncased",
        "representation": "Transformer encoder",
        "training_target": "annotator vote fractions / binomial NLL",
        "binary_target_for_headline": "majority scheme",
        "thresholding": "per-label validation-tuned thresholds",
        "source": "Code/BERT_models/bert_large_uncased_random_seed42_metrics.json",
    },
    {
        "family": "BERT",
        "model": "ModernBERT-base",
        "representation": "Transformer encoder",
        "training_target": "annotator vote fractions / binomial NLL",
        "binary_target_for_headline": "majority scheme",
        "thresholding": "per-label validation-tuned thresholds",
        "source": "Code/BERT_models/modernbert_base_random_seed42_metrics.json",
    },
])

model_setup


In [ ]:
def load_bert_summary():
    rows = []
    for path in sorted(BERT_DIR.glob("*_random_seed42_metrics.json")):
        metrics = json.loads(path.read_text())
        binary = metrics["binary_scheme_metrics"]["majority"]["tuned_binary_metrics"]
        soft = metrics["soft_summary"]
        rows.append({
            "family": "BERT",
            "model_id": metrics["experiment_name"],
            "model_name": metrics["model_name"],
            "comparison_target": "majority / validation-tuned",
            "micro_f1": binary["micro_f1"],
            "macro_f1": binary["macro_f1"],
            "samples_f1": binary["samples_f1"],
            "hamming_loss": binary["hamming_loss"],
            "macro_mae": soft["macro_mae"],
            "mean_pearson": soft["mean_pearson"],
            "mean_spearman": soft["mean_spearman"],
        })
    return pd.DataFrame(rows)


def load_tfidf_summary():
    path = TFIDF_OUTPUT_DIR / "tfidf_model_summary.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run Code/TF_IDF/TFIDF_Ridge_Baseline.ipynb through "
            "the 'Save TF-IDF Outputs' section first."
        )
    return pd.read_csv(path)

summary = pd.concat([load_tfidf_summary(), load_bert_summary()], ignore_index=True)
summary = summary.sort_values("macro_f1", ascending=False)

summary[[
    "family", "model_name", "comparison_target", "micro_f1", "macro_f1",
    "samples_f1", "hamming_loss", "macro_mae", "mean_pearson"
]].style.format({
    "micro_f1": "{:.3f}",
    "macro_f1": "{:.3f}",
    "samples_f1": "{:.3f}",
    "hamming_loss": "{:.3f}",
    "macro_mae": "{:.3f}",
    "mean_pearson": "{:.3f}",
})

In [ ]:
best_macro = summary.loc[summary["macro_f1"].idxmax()]
best_micro = summary.loc[summary["micro_f1"].idxmax()]
best_samples = summary.loc[summary["samples_f1"].idxmax()]
best_hamming = summary.loc[summary["hamming_loss"].idxmin()]

print(f"Best macro F1: {best_macro['model_name']} ({best_macro['macro_f1']:.3f})")
print(f"Best micro F1: {best_micro['model_name']} ({best_micro['micro_f1']:.3f})")
print(f"Best samples F1: {best_samples['model_name']} ({best_samples['samples_f1']:.3f})")
print(f"Lowest hamming loss: {best_hamming['model_name']} ({best_hamming['hamming_loss']:.3f})")

In [ ]:
plot_df = summary.sort_values("macro_f1", ascending=True)
metrics = ["macro_f1", "micro_f1", "samples_f1"]
fig, axes = plt.subplots(1, len(metrics), figsize=(15, 5), sharey=True)

for ax, metric in zip(axes, metrics):
    colors = plot_df["family"].map({"BERT": "lightblue", "TF-IDF": "orange", "Prevalence": "grey"})
    ax.barh(plot_df["model_name"], plot_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_xlim(0, 0.7)
    ax.grid(axis="x", alpha=0.25)
    for i, value in enumerate(plot_df[metric]):
        ax.text(value + 0.01, i, f"{value:.3f}", va="center", fontsize=9)

fig.suptitle("Binary Test Performance")
plt.tight_layout()
plt.show()

BERT was evaluated under three binary target definitions. Use majority and binary setup: weak_presence and strong_presence.

In [ ]:
bert_scheme_summary = pd.read_csv(BERT_DIR / "random_splits_binomial_nll_by_model.csv")
bert_scheme_summary[[
    "experiment_name", "macro_mae_mean", "mean_pearson_mean",
    "weak_presence_tuned_macro_f1_mean", "majority_tuned_macro_f1_mean",
    "strong_presence_tuned_macro_f1_mean", "n_runs"
]].sort_values("majority_tuned_macro_f1_mean", ascending=False).style.format({
    "macro_mae_mean": "{:.3f}",
    "mean_pearson_mean": "{:.3f}",
    "weak_presence_tuned_macro_f1_mean": "{:.3f}",
    "majority_tuned_macro_f1_mean": "{:.3f}",
    "strong_presence_tuned_macro_f1_mean": "{:.3f}",
})

In [ ]:
scheme_plot = bert_scheme_summary.set_index("experiment_name")[[
    "weak_presence_tuned_macro_f1_mean",
    "majority_tuned_macro_f1_mean",
    "strong_presence_tuned_macro_f1_mean",
]]
scheme_plot.columns = ["weak_presence", "majority", "strong_presence"]

ax = scheme_plot.plot(kind="bar", figsize=(9, 4), color=["green", "lightblue", "red"])
ax.set_title("BERT Macro F1 Changes With Target Definition")
ax.set_ylabel("Macro F1")
ax.set_xlabel("Model")
ax.set_ylim(0, 0.75)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

Comparing per-label F1 for the strongest TF-IDF variants and BERT-large under the comparable majority-style binary task.

But Abberation only has 2 positive test examples in the TF-IDF report Humour has only 3. 

In [ ]:
def parse_classification_report(report, model_name):
    rows = []
    pattern = re.compile(r"^\s*(GM|PR|A|EJ|N|UC|V|H)\s+(\d+\.\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)\s+(\d+)\s*$")
    for line in report.splitlines():
        match = pattern.match(line)
        if match:
            label, precision, recall, f1, support = match.groups()
            rows.append({
                "model_name": model_name,
                "label": label,
                "strategy": STRATEGY_NAMES[label],
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "support": int(support),
            })
    return rows


tfidf_per_label = pd.read_csv(TFIDF_OUTPUT_DIR / "tfidf_per_label_metrics.csv")
tfidf_per_label = tfidf_per_label[
    tfidf_per_label["model_id"].isin(["tfidf_tuned_ridge"])
].copy()
tfidf_per_label["strategy"] = tfidf_per_label["label"].map(STRATEGY_NAMES)

bert_large_metrics = json.loads((BERT_DIR / "bert_large_uncased_random_seed42_metrics.json").read_text())
bert_large_report = bert_large_metrics["binary_scheme_metrics"]["majority"]["tuned_classification_report"]
bert_large_per_label = pd.DataFrame(parse_classification_report(bert_large_report, "BERT-large uncased"))

per_label_f1 = pd.concat([
    tfidf_per_label[["model_name", "label", "strategy", "precision", "recall", "f1", "support"]],
    bert_large_per_label,
], ignore_index=True)

per_label_wide = per_label_f1.pivot_table(
    index=["label", "strategy", "support"],
    columns="model_name",
    values="f1",
).reset_index()

per_label_wide["best_model_for_label"] = per_label_wide[[
    "TF-IDF + tuned Ridge", "BERT-large uncased"
]].idxmax(axis=1)

per_label_wide.style.format({
    "TF-IDF + tuned Ridge": "{:.2f}",
    "BERT-large uncased": "{:.2f}",
})


In [ ]:
plot_label_df = per_label_f1.pivot(index="label", columns="model_name", values="f1").loc[LABELS]
ax = plot_label_df.plot(kind="bar", figsize=(11, 4), color=["green", "lightblue", "red"])
ax.set_title("Per-Label Test F1")
ax.set_ylabel("F1")
ax.set_xlabel("Label")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.legend(title="Model", loc="upper right")
plt.tight_layout()
plt.show()


- best model for binary comparison bert-large-uncased  macro F1 (0.460) and micro F1 (0.584)
- Ridge vs prevelance baseline reaches macro F1 (0.400) and micro F1 (0.538), compared with the prevalence baseline macro F1 (0.084) and micro F1 (0.413)
- BERT-large also has the strongest soft-label for BERT models with the lowest macro MAE (0.155) and highest mean Pearson correlation (0.539).
- ModernBERT-base is strongest on weak_presence macro F1, but BERT-large is strongest on majority and strong_presence`
- Frequent labels are easier PR and GM are stronger. H and A are unstable or weak because of low number
